# 🚀 DataMate — Zalo AI Challenge 2026: SFT & GRPO Training Pipeline

Notebook này hướng dẫn quy trình huấn luyện nâng cao cho mô hình **Qwen2.5-Coder-7B-Instruct** dựa trên tập dữ liệu `train` (30 doanh nghiệp, 3,000 câu hỏi có sẵn đáp án `answers.jsonl`):

1. **Pre-build Cache:** Tạo cơ sở dữ liệu DuckDB tốc độ cao cho toàn bộ tập `train`.
2. **Rejection Sampling (RFT):** Tự động sinh code (kèm cơ chế tự sửa lỗi 2 lượt) và lọc các mẫu đúng 100% để tạo tập dữ liệu vàng (`train/sft_dataset.jsonl`).
3. **Supervised Fine-Tuning (SFT LoRA):** Huấn luyện mô hình nắm vững cú pháp DuckDB tiếng Việt (Completion-only Loss Masking).
4. **GRPO Reinforcement Learning:** Huấn luyện học tăng cường tự sửa sai với hàm thưởng khách quan (Rule-based Verifiable Outcome Reward).
5. **Suy luận & Nộp bài:** Chạy mô hình đã fine-tune trên `public_test` để xuất file nộp bài chuẩn.

## 1. Kiểm tra thông tin GPU & Cài đặt môi trường

In [ ]:
# 1.1 Kiểm tra GPU (Khuyến nghị dùng A100 GPU trên Colab Pro)
!nvidia-smi

In [ ]:
# 1.2 Cài đặt các thư viện cần thiết
!pip install -q -U transformers accelerate datasets peft 'trl>=0.14.0' bitsandbytes duckdb unidecode tqdm python-calamine

## 2. Kết nối Google Drive & Đồng bộ mã nguồn (Branch dev)

In [ ]:
# 2.1 Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ZALO_AI_Challenge'
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd {PROJECT_DIR}

In [ ]:
# 2.2 Đồng bộ code mới nhất từ GitHub nhánh dev
import os
if not os.path.exists('.git'):
    !git init
    !git remote add origin https://github.com/JimmyK300/zalo-vecna.git

!git fetch origin dev && git reset --hard origin/dev
!git status

## 3. Tiền xử lý DuckDB Cache cho Tập `train`

In [ ]:
# ⚡ 3.1 Build cache DuckDB cho tập train (nếu đã chạy trước đó thì KHÔNG CẦN chạy lại, code mới sẽ tự động sửa cache trong 0.01s!)
!python prebuild_cache.py --data_dir train

## 4. Rejection Sampling (RFT) — Khai thác dữ liệu vàng (3,000 câu)

In [ ]:
# ⛏️ 4.1 Khai thác các mẫu đúng 100% trên toàn bộ 3,000 câu train
# (Tự động sửa cache DuckDB đã tạo trước đó + thử lại thông minh 2 lượt/câu)
!python build_training_dataset.py \
    --data_dir train \
    --output train/sft_dataset.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --samples_per_question 2 \
    --batch_size 16 \
    --fp16 \
    --overwrite

# Đếm số lượng mẫu vàng đã khai thác thành công
!wc -l train/sft_dataset.jsonl

## 5. Supervised Fine-Tuning (SFT với LoRA)

In [ ]:
# 🧠 5.1 Huấn luyện LoRA SFT trên tập dữ liệu vàng (mất ~20-30 phút trên A100)
!python train_sft.py \
    --data_path train/sft_dataset.jsonl \
    --output_dir checkpoints/sft_adapter \
    --epochs 3 \
    --batch_size 2 \
    --grad_accum 8 \
    --max_seq_length 4096 \
    --lr 2e-4 \
    --bf16

## 6. Group Relative Policy Optimization (GRPO)

In [ ]:
# 🎯 6.1 Huấn luyện học tăng cường GRPO với DuckDB Verifiable Reward (Warm-start từ SFT)
!python train_grpo.py \
    --data_dir train \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/sft_adapter \
    --output_dir checkpoints/grpo_adapter \
    --num_generations 4 \
    --epochs 1 \
    --batch_size 4 \
    --grad_accum 2 \
    --lr 5e-6 \
    --bf16 \
    --limit 500

## 7. Đánh giá & Suy luận trên `public_test` với Model đã Fine-tune

In [ ]:
# 🚀 7.1 Chạy suy luận Public Test với LoRA Adapter đã huấn luyện (ưu tiên grpo_adapter)
!python run_public_test.py \
    --data_dir public_test \
    --output submission_finetuned.jsonl \
    --model_id Qwen/Qwen2.5-Coder-7B-Instruct \
    --adapter_path checkpoints/grpo_adapter \
    --batch_size 16 \
    --fp16 \
    --overwrite

# Xem thử 10 câu đầu tiên
!head -n 10 submission_finetuned.jsonl

## 8. 🌙 Chế độ Treo Máy Qua Đêm (Chạy liên hoàn RFT -> SFT -> GRPO -> Public Test)

In [ ]:
# 🌙 8.1 Một cú click chạy trọn gói từ đầu đến cuối để treo máy qua đêm
%cd /content/drive/MyDrive/ZALO_AI_Challenge
!git fetch origin dev && git reset --hard origin/dev
!python build_training_dataset.py --data_dir train --output train/sft_dataset.jsonl --model_id Qwen/Qwen2.5-Coder-7B-Instruct --samples_per_question 2 --batch_size 16 --fp16 --overwrite && \
 python train_sft.py --data_path train/sft_dataset.jsonl --output_dir checkpoints/sft_adapter --epochs 3 --batch_size 2 --grad_accum 8 --max_seq_length 4096 --lr 2e-4 --bf16 && \
 python train_grpo.py --data_dir train --model_id Qwen/Qwen2.5-Coder-7B-Instruct --adapter_path checkpoints/sft_adapter --output_dir checkpoints/grpo_adapter --num_generations 4 --epochs 1 --batch_size 4 --grad_accum 2 --lr 5e-6 --bf16 --limit 500 && \
 python run_public_test.py --data_dir public_test --output submission_finetuned.jsonl --model_id Qwen/Qwen2.5-Coder-7B-Instruct --adapter_path checkpoints/grpo_adapter --batch_size 16 --fp16 --overwrite